# PatchTST Re-implementation — Full Table 3 Reproduction
Reproduces the multivariate long-term forecasting results from "A Time Series is Worth 64 Words" (Nie et al., ICLR 2023), Table 3 (PatchTST/42, look-back = 336).

**Datasets covered:** ETTh1, ETTh2, ETTm1, ETTm2, Weather, Electricity, Traffic, ILI.

**Before running:** Runtime → Change runtime type → T4 GPU. Full sweep is 8 datasets × 4 horizons = **32 training runs** (the heavy ones are Electricity and Traffic). Comment out entries in `DATASETS` below to subset.

## 0. Setup

In [ ]:
import os

GITHUB_URL = "https://github.com/mmichellezhou/patchtst.git"

if os.path.exists("/content/patchtst"):
    !git -C /content/patchtst pull
else:
    !git clone {GITHUB_URL} /content/patchtst

# ETT (4 CSVs from a separate repo)
if not os.path.exists("/content/patchtst/data/ETDataset/ETT-small/ETTh1.csv"):
    !rm -rf /content/patchtst/data/ETDataset
    !git clone https://github.com/zhouhaoyi/ETDataset.git /content/patchtst/data/ETDataset

# Weather, Electricity, Traffic, ILI (Hugging Face mirror of thuml/Time-Series-Library)
HF_BASE = "https://huggingface.co/datasets/thuml/Time-Series-Library/resolve/main"
DATA_DIR = "/content/patchtst/data"
for remote, local in [
    ("weather/weather.csv",                 "weather.csv"),
    ("electricity/electricity.csv",         "electricity.csv"),
    ("traffic/traffic.csv",                 "traffic.csv"),
    ("illness/national_illness.csv",        "national_illness.csv"),
]:
    target = f"{DATA_DIR}/{local}"
    if not os.path.exists(target):
        !curl -sL "{HF_BASE}/{remote}" -o "{target}"
    print(f"{local}: {os.path.getsize(target)/1e6:.1f} MB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys, os

PROJ_ROOT   = "/content/patchtst"
CODE_DIR    = f"{PROJ_ROOT}/code"
RESULTS_DIR = "/content/drive/MyDrive/final project/results"
FIGURES_DIR = f"{RESULTS_DIR}/figures"
TABLES_DIR  = f"{RESULTS_DIR}/tables"
CKPT_DIR    = f"{RESULTS_DIR}/checkpoints"

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

for d in [FIGURES_DIR, TABLES_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Code:",    CODE_DIR)
print("Results:", RESULTS_DIR)

In [ ]:
!pip install -q -r "/content/patchtst/requirements.txt"

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from config import Config
from train import train
from evaluate import load_and_evaluate

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 1. Datasets and Paper Reference Results

Per-dataset prediction horizons and Table 3 (PatchTST/42, look-back = 336) reference values.
ILI uses different horizons {24, 36, 48, 60} and `seq_len = 104` per the official scripts.

In [ ]:
# Standard horizons for everything except ILI
STD_PRED_LENS = [96, 192, 336, 720]
ILI_PRED_LENS = [24, 36, 48, 60]

# Per-dataset (pred_lens, Table-3 PatchTST/42 reference, look-back L=336).
# Verified against Nie et al. ICLR 2023, Table 3.
DATASETS = {
    "ETTh1":       (STD_PRED_LENS, {96: (0.375, 0.399), 192: (0.414, 0.421), 336: (0.431, 0.436), 720: (0.449, 0.466)}),
    "ETTh2":       (STD_PRED_LENS, {96: (0.274, 0.336), 192: (0.339, 0.379), 336: (0.331, 0.380), 720: (0.379, 0.422)}),
    "ETTm1":       (STD_PRED_LENS, {96: (0.290, 0.342), 192: (0.332, 0.369), 336: (0.366, 0.392), 720: (0.420, 0.424)}),
    "ETTm2":       (STD_PRED_LENS, {96: (0.165, 0.255), 192: (0.220, 0.292), 336: (0.278, 0.329), 720: (0.367, 0.385)}),
    "weather":     (STD_PRED_LENS, {96: (0.152, 0.199), 192: (0.197, 0.243), 336: (0.249, 0.283), 720: (0.320, 0.335)}),
    "electricity": (STD_PRED_LENS, {96: (0.130, 0.222), 192: (0.148, 0.240), 336: (0.167, 0.261), 720: (0.202, 0.291)}),
    "traffic":     (STD_PRED_LENS, {96: (0.367, 0.251), 192: (0.385, 0.259), 336: (0.398, 0.265), 720: (0.434, 0.287)}),
    "ili":         (ILI_PRED_LENS, {24: (1.522, 0.814), 36: (1.430, 0.834), 48: (1.673, 0.854), 60: (1.529, 0.862)}),
}

print(f"{len(DATASETS)} datasets, total runs: {sum(len(p) for p, _ in DATASETS.values())}")

## 2. Train and Evaluate All (dataset × pred_len)

For each dataset we instantiate `Config(dataset_name=..., pred_len=...)` — per-dataset hyperparameters (d_model, n_heads, batch_size, lr, seq_len, patch_len/stride) are pulled from `_DATASET_SPECS` in `config.py`, matching the official PatchTST/42 scripts.

In [ ]:
all_results    = {}   # {dataset_name: {pred_len: {"MSE": float, "MAE": float}}}
loss_histories = {}   # {dataset_name: {pred_len: {"train": [...], "val": [...]}}}

for dataset_name, (pred_lens, _) in DATASETS.items():
    all_results[dataset_name]    = {}
    loss_histories[dataset_name] = {}

    for pred_len in pred_lens:
        tag = f"{dataset_name}_pl{pred_len}"
        print(f"\n{'='*60}\n  {tag}\n{'='*60}")

        config = Config(dataset_name=dataset_name, pred_len=pred_len)
        config.checkpoint_path = f"{CKPT_DIR}/{tag}"
        config.save_path       = f"{RESULTS_DIR}/{tag}"
        os.makedirs(config.checkpoint_path, exist_ok=True)
        os.makedirs(config.save_path,       exist_ok=True)

        # Skip if already trained (idempotent re-runs)
        ckpt = f"{config.checkpoint_path}/best_model.pt"
        if os.path.exists(ckpt):
            print(f"  [skip training] checkpoint exists at {ckpt}")
            train_losses, val_losses = [], []
        else:
            train_losses, val_losses = train(config)

        loss_histories[dataset_name][pred_len] = {"train": train_losses, "val": val_losses}

        mse, mae = load_and_evaluate(config, device=DEVICE)
        all_results[dataset_name][pred_len] = {"MSE": round(mse, 3), "MAE": round(mae, 3)}
        print(f"  Test  MSE={mse:.3f}  MAE={mae:.3f}")

## 3. Per-Dataset Comparison Tables

In [ ]:
def comparison_df(dataset_name):
    pred_lens, paper = DATASETS[dataset_name]
    ours = all_results[dataset_name]
    rows = []
    for pl in pred_lens:
        p_mse, p_mae = paper[pl]
        rows.append({
            "pred_len":   pl,
            "Ours MSE":   ours[pl]["MSE"],
            "Paper MSE":  p_mse,
            "Ours MAE":   ours[pl]["MAE"],
            "Paper MAE":  p_mae,
        })
    df = pd.DataFrame(rows).set_index("pred_len")
    return df

for ds in DATASETS:
    print(f"\n=== {ds} ===")
    df = comparison_df(ds)
    df.to_csv(f"{TABLES_DIR}/{ds}_results.csv")
    display(df)

## 4. Summary: Average MSE / MAE per Dataset (Ours vs Paper)

In [ ]:
summary_rows = []
for ds, (pred_lens, paper) in DATASETS.items():
    ours_mse = np.mean([all_results[ds][pl]["MSE"] for pl in pred_lens])
    ours_mae = np.mean([all_results[ds][pl]["MAE"] for pl in pred_lens])
    pap_mse  = np.mean([paper[pl][0] for pl in pred_lens])
    pap_mae  = np.mean([paper[pl][1] for pl in pred_lens])
    summary_rows.append({
        "Dataset":   ds,
        "Ours MSE":  round(ours_mse, 3),
        "Paper MSE": round(pap_mse,  3),
        "Ours MAE":  round(ours_mae, 3),
        "Paper MAE": round(pap_mae,  3),
    })

summary = pd.DataFrame(summary_rows).set_index("Dataset")
summary.to_csv(f"{TABLES_DIR}/summary.csv")
display(summary)

## 5. Per-Dataset MSE Comparison Plots

In [ ]:
n_ds = len(DATASETS)
fig, axes = plt.subplots((n_ds + 1) // 2, 2, figsize=(13, 3.2 * ((n_ds + 1) // 2)))
axes = axes.flatten()

for ax, (ds, (pred_lens, paper)) in zip(axes, DATASETS.items()):
    x = range(len(pred_lens))
    width = 0.35
    our_mse   = [all_results[ds][pl]["MSE"] for pl in pred_lens]
    paper_mse = [paper[pl][0]               for pl in pred_lens]
    ax.bar([i - width/2 for i in x], our_mse,   width, label="Ours",  color="steelblue")
    ax.bar([i + width/2 for i in x], paper_mse, width, label="Paper", color="tomato", alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(pred_lens)
    ax.set_xlabel("Prediction horizon")
    ax.set_ylabel("MSE")
    ax.set_title(ds)
    ax.legend(fontsize=8)

# Hide any unused axes
for ax in axes[len(DATASETS):]:
    ax.set_visible(False)

fig.suptitle("PatchTST/42 — Ours vs Paper (MSE)", fontsize=13)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/all_datasets_mse.png", dpi=150)
plt.show()

## 6. Loss Curves (per dataset)

In [ ]:
for ds, (pred_lens, _) in DATASETS.items():
    fig, axes = plt.subplots(1, len(pred_lens), figsize=(4 * len(pred_lens), 3.2))
    if len(pred_lens) == 1:
        axes = [axes]
    for ax, pl in zip(axes, pred_lens):
        h = loss_histories[ds][pl]
        if not h["train"]:
            ax.text(0.5, 0.5, "skipped\n(checkpoint reused)", ha="center", va="center", transform=ax.transAxes, fontsize=9, color="gray")
        else:
            ax.plot(h["train"], label="train")
            ax.plot(h["val"],   label="val")
            ax.legend(fontsize=8)
        ax.set_title(f"pred_len = {pl}")
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE")
    fig.suptitle(f"{ds} — training loss", fontsize=12)
    fig.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/{ds}_loss_curves.png", dpi=150)
    plt.show()

## 7. Poster Figures

Three forecast-visualization figures, each saved as a high-DPI PNG to `FIGURES_DIR`. Each cell loads its own checkpoint(s) and produces one self-contained image. Run only after section 2 (training) is complete.

### Figure 2 — Per-Dataset Forecast Grid

One panel per dataset at horizon T=96 (T=24 for ILI). Last channel plotted: **input** (blue solid) → **ground truth** (blue dashed) → **PatchTST prediction** (red). The vertical dotted line marks the forecast boundary.

The point is that one architecture, with no per-dataset retuning beyond the standard hyperparameters, generalizes across 8 very different domains.

In [ ]:
from patchtst import PatchTST
from dataset import get_dataloaders

# Figure 2: Per-dataset forecast grid (2x4).
# pred_len = 96 for everything (= 24 for ILI since 96 isn't in its horizon set).
GRID_PRED_LEN = {ds: (24 if ds == "ili" else 96) for ds in DATASETS}

fig, axes = plt.subplots(2, 4, figsize=(18, 6))
axes = axes.flatten()

for ax, ds in zip(axes, DATASETS):
    pl = GRID_PRED_LEN[ds]
    config = Config(dataset_name=ds, pred_len=pl)
    config.checkpoint_path = f"{CKPT_DIR}/{ds}_pl{pl}"

    df_head = pd.read_csv(config.data_path, nrows=1)
    cols = [c for c in df_head.columns if c != "date"]
    ch_idx = -1                 # last channel: OT for ETT, last col elsewhere
    ch_name = cols[ch_idx]

    _, _, test_loader = get_dataloaders(config)
    model = PatchTST(config).to(DEVICE)
    model.load_state_dict(torch.load(f"{config.checkpoint_path}/best_model.pt", map_location=DEVICE))
    model.eval()

    x_batch, y_batch = next(iter(test_loader))
    with torch.no_grad():
        pred = model(x_batch.to(DEVICE)).cpu()

    seq_len, pred_len = x_batch.shape[1], y_batch.shape[1]
    t_in  = np.arange(seq_len)
    t_out = np.arange(seq_len, seq_len + pred_len)

    ax.plot(t_in,  x_batch[0, :, ch_idx].numpy(), color="steelblue", linewidth=1.0, label="input")
    ax.plot(t_out, y_batch[0, :, ch_idx].numpy(), color="steelblue", linewidth=1.2, linestyle="--", label="truth")
    ax.plot(t_out, pred[0,    :, ch_idx].numpy(), color="tomato",    linewidth=1.5, label="prediction")
    ax.axvline(seq_len, color="gray", linestyle=":", alpha=0.5)
    ax.set_title(f"{ds}  (T={pred_len}, ch={ch_name})", fontsize=10)
    ax.tick_params(labelsize=8)

axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("PatchTST forecasts across 8 benchmarks", fontsize=14)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/poster_forecast_grid.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 3 — Forecast Degradation Across Horizons

Same Traffic input window, predicted by **4 separately-trained models** (one per horizon T ∈ {96, 192, 336, 720}). Each prediction is a single forward pass — not autoregressive rollout.

Predictions stay sharp at T=96 and soften (lose high-frequency structure) as the horizon grows, matching the paper's MSE growth from 0.367 → 0.434.

In [ ]:
# Figure 3: Multi-horizon degradation on Traffic (1x4).
# Same input window, 4 different prediction horizons (each from its own checkpoint).
DEGR_DATASET = "traffic"
DEGR_HORIZONS = [96, 192, 336, 720]
DEGR_CHANNEL = 0

fig, axes = plt.subplots(1, 4, figsize=(20, 3.6), sharey=True)

for ax, pl in zip(axes, DEGR_HORIZONS):
    config = Config(dataset_name=DEGR_DATASET, pred_len=pl)
    config.checkpoint_path = f"{CKPT_DIR}/{DEGR_DATASET}_pl{pl}"

    _, _, test_loader = get_dataloaders(config)
    model = PatchTST(config).to(DEVICE)
    model.load_state_dict(torch.load(f"{config.checkpoint_path}/best_model.pt", map_location=DEVICE))
    model.eval()

    x_batch, y_batch = next(iter(test_loader))
    with torch.no_grad():
        pred = model(x_batch.to(DEVICE)).cpu()

    seq_len, pred_len = x_batch.shape[1], y_batch.shape[1]
    t_in  = np.arange(seq_len)
    t_out = np.arange(seq_len, seq_len + pred_len)

    ax.plot(t_in,  x_batch[0, :, DEGR_CHANNEL].numpy(), color="steelblue", linewidth=1.0, label="input")
    ax.plot(t_out, y_batch[0, :, DEGR_CHANNEL].numpy(), color="steelblue", linewidth=1.2, linestyle="--", label="truth")
    ax.plot(t_out, pred[0,    :, DEGR_CHANNEL].numpy(), color="tomato",    linewidth=1.5, label="prediction")
    ax.axvline(seq_len, color="gray", linestyle=":", alpha=0.5)
    ax.set_title(f"horizon T = {pl}", fontsize=11)
    ax.set_xlabel("Time step", fontsize=9)
    ax.tick_params(labelsize=8)

axes[0].set_ylabel(f"{DEGR_DATASET} channel {DEGR_CHANNEL}", fontsize=9)
axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle(f"Forecast quality across prediction horizons — {DEGR_DATASET}", fontsize=13)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/poster_horizons_{DEGR_DATASET}.png", dpi=200, bbox_inches="tight")
plt.show()

### Figure 4 — Channel-Independence on Traffic

8 of Traffic's 862 sensor channels, all predicted in **one forward pass** by a single PatchTST. Transformer weights are shared across channels but each channel's series is processed independently — there is no cross-channel attention.

Wildly different sensor patterns (different highway locations, different traffic dynamics) handled by one set of weights — the paper's central architectural claim made visual.

In [ ]:
# Figure 4: Channel-independence on Traffic (4x2).
# 8 of 862 traffic channels predicted by one shared model in a single forward pass.
CH_DATASET = "traffic"
CH_PRED_LEN = 96
# Spread channel indices across the full 862. Hand-pick if you want more striking patterns.
CH_INDICES = [0, 100, 200, 300, 400, 550, 700, 850]

config = Config(dataset_name=CH_DATASET, pred_len=CH_PRED_LEN)
config.checkpoint_path = f"{CKPT_DIR}/{CH_DATASET}_pl{CH_PRED_LEN}"

_, _, test_loader = get_dataloaders(config)
model = PatchTST(config).to(DEVICE)
model.load_state_dict(torch.load(f"{config.checkpoint_path}/best_model.pt", map_location=DEVICE))
model.eval()

x_batch, y_batch = next(iter(test_loader))
with torch.no_grad():
    pred = model(x_batch.to(DEVICE)).cpu()

seq_len, pred_len = x_batch.shape[1], y_batch.shape[1]
t_in  = np.arange(seq_len)
t_out = np.arange(seq_len, seq_len + pred_len)

fig, axes = plt.subplots(4, 2, figsize=(14, 9), sharex=True)
axes = axes.flatten()
for ax, ch_idx in zip(axes, CH_INDICES):
    ax.plot(t_in,  x_batch[0, :, ch_idx].numpy(), color="steelblue", linewidth=0.9, label="input")
    ax.plot(t_out, y_batch[0, :, ch_idx].numpy(), color="steelblue", linewidth=1.2, linestyle="--", label="truth")
    ax.plot(t_out, pred[0,    :, ch_idx].numpy(), color="tomato",    linewidth=1.4, label="prediction")
    ax.axvline(seq_len, color="gray", linestyle=":", alpha=0.5)
    ax.set_title(f"channel {ch_idx}", fontsize=10)
    ax.tick_params(labelsize=8)

axes[0].legend(loc="upper left", fontsize=8)
for ax in axes[-2:]:
    ax.set_xlabel("Time step", fontsize=9)
fig.suptitle(f"Channel-independence: 8 of 862 traffic channels, one shared model", fontsize=13)
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/poster_channels_{CH_DATASET}.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Ablation — Multi-Scale Patches

PatchTST/42 uses a single patch length (16). Our extension lets the model see the same series at multiple patch resolutions simultaneously (default `(8, 16, 32)`), so short patches catch fine-grained structure and long patches catch slower trends.

This section trains a multi-scale variant on a small subset and compares against the standard run from section 2. Standard checkpoints are reused — only the multi-scale variant retrains. Adjust `ABLATION_DATASETS` / `ABLATION_PRED_LENS` to widen or narrow the sweep.

In [ ]:
# Subset for the ablation. Defaults: 3 datasets x 2 horizons = 6 multi-scale runs to train.
# Standard runs are reused from section 2 (no retraining).
ABLATION_DATASETS  = ["ETTh1", "weather", "electricity"]
ABLATION_PRED_LENS = [96, 720]
PATCH_SCALES       = (8, 16, 32)

ablation_results = {}   # {dataset: {pred_len: {"std": (mse, mae), "multi": (mse, mae)}}}

for ds in ABLATION_DATASETS:
    ablation_results[ds] = {}
    for pl in ABLATION_PRED_LENS:
        print(f"\n--- {ds}  T={pl} ---")

        # Standard variant: reuse section-2 checkpoint
        std_cfg = Config(dataset_name=ds, pred_len=pl)
        std_cfg.checkpoint_path = f"{CKPT_DIR}/{ds}_pl{pl}"
        if not os.path.exists(f"{std_cfg.checkpoint_path}/best_model.pt"):
            print(f"  (no standard checkpoint — training)")
            train(std_cfg)
        std_mse, std_mae = load_and_evaluate(std_cfg, device=DEVICE)
        print(f"  standard:    MSE={std_mse:.3f}  MAE={std_mae:.3f}")

        # Multi-scale variant: separate checkpoint dir
        ms_cfg = Config(
            dataset_name=ds, pred_len=pl,
            use_multiscale_patches=True, patch_scales=PATCH_SCALES,
        )
        ms_cfg.checkpoint_path = f"{CKPT_DIR}/{ds}_pl{pl}_multiscale"
        ms_cfg.save_path       = f"{RESULTS_DIR}/{ds}_pl{pl}_multiscale"
        os.makedirs(ms_cfg.checkpoint_path, exist_ok=True)
        os.makedirs(ms_cfg.save_path,       exist_ok=True)
        if not os.path.exists(f"{ms_cfg.checkpoint_path}/best_model.pt"):
            train(ms_cfg)
        ms_mse, ms_mae = load_and_evaluate(ms_cfg, device=DEVICE)
        print(f"  multi-scale: MSE={ms_mse:.3f}  MAE={ms_mae:.3f}")

        ablation_results[ds][pl] = {"std": (std_mse, std_mae), "multi": (ms_mse, ms_mae)}

### Comparison Table

`ΔMSE %` and `ΔMAE %` are positive when multi-scale **hurts** and negative when it helps.

In [ ]:
rows = []
for ds, by_pl in ablation_results.items():
    for pl, vals in by_pl.items():
        std_mse, std_mae = vals["std"]
        ms_mse,  ms_mae  = vals["multi"]
        rows.append({
            "Dataset":    ds,
            "T":          pl,
            "Std MSE":    round(std_mse, 3),
            "Multi MSE":  round(ms_mse,  3),
            "ΔMSE %":     round(100 * (ms_mse - std_mse) / std_mse, 1),
            "Std MAE":    round(std_mae, 3),
            "Multi MAE":  round(ms_mae,  3),
            "ΔMAE %":     round(100 * (ms_mae - std_mae) / std_mae, 1),
        })

ablation_df = pd.DataFrame(rows).set_index(["Dataset", "T"])
ablation_df.to_csv(f"{TABLES_DIR}/multiscale_ablation.csv")
display(ablation_df)

### Bar Chart

Side-by-side MSE for each `(dataset, horizon)` pair. Bar height is MSE, so shorter is better.

In [ ]:
labels  = [f"{ds}\nT={pl}" for ds in ABLATION_DATASETS for pl in ABLATION_PRED_LENS]
std_mse = [ablation_results[ds][pl]["std"][0]   for ds in ABLATION_DATASETS for pl in ABLATION_PRED_LENS]
ms_mse  = [ablation_results[ds][pl]["multi"][0] for ds in ABLATION_DATASETS for pl in ABLATION_PRED_LENS]

x = np.arange(len(labels))
width = 0.38

fig, ax = plt.subplots(figsize=(max(8, len(labels) * 1.4), 4.6))
ax.bar(x - width/2, std_mse, width, label="Standard PatchTST",                     color="steelblue")
ax.bar(x + width/2, ms_mse,  width, label=f"Multi-scale  scales={PATCH_SCALES}",   color="darkorange")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("MSE")
ax.set_title("Multi-scale patches vs single-scale (lower is better)")
ax.legend()
fig.tight_layout()
fig.savefig(f"{FIGURES_DIR}/multiscale_ablation.png", dpi=200, bbox_inches="tight")
plt.show()